# Multiple linear regression: two inputs, a plane in 3D

Drag the 3D plot to turn it.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [2]:
# 100 rows, 2 input columns, noise added; random_state fixes the data
X, y = make_regression(n_samples=100, n_features=2, n_informative=2, noise=50, random_state=7)
df = pd.DataFrame({"feature1": X[:, 0], "feature2": X[:, 1], "target": y})
df.head()

,feature1,feature2,target
0,0.269412,-0.524605,32.578326
1,-1.434963,0.501624,-75.287305
2,-0.099611,1.699537,56.171893
3,-0.281108,-1.536102,-97.012302
4,-0.193856,0.697294,-103.355634


In [3]:
px.scatter_3d(df, x="feature1", y="feature2", z="target")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)
lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)
print("MAE", mean_absolute_error(y_test, y_pred))
print("MSE", mean_squared_error(y_test, y_pred))
print("R2 ", r2_score(y_test, y_pred))
print("coefficients", lr.coef_, "intercept", lr.intercept_)

MAE 40.069214173122575
MSE 2614.8644648646195
R2  0.6061984594164702
coefficients [58.60569036 29.1140461 ] intercept -1.9011250399840933


## The fitted plane

In [5]:
g = np.linspace(-3, 3, 20)
G1, G2 = np.meshgrid(g, g)
# predict on every grid point, then reshape back into a 20 x 20 surface
Z = lr.predict(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)
fig = px.scatter_3d(df, x="feature1", y="feature2", z="target")
fig.add_trace(go.Surface(x=G1, y=G2, z=Z, opacity=0.5, showscale=False))
fig

## Check: how the noise limits R²

In [6]:
# the same data recipe with less noise; only the noise changes
from sklearn.metrics import r2_score
for noise in [50, 25, 0]:
    Xn, yn = make_regression(n_samples=100, n_features=2, n_informative=2, noise=noise, random_state=7)
    a, b, c, d = train_test_split(Xn, yn, test_size=0.2, random_state=3)
    print(noise, round(r2_score(d, LinearRegression().fit(a, c).predict(b)), 3))   # 0.606, 0.852, 1.0
print('input std:', X.std(axis=0).round(2))   # both about 1: same scale


50 0.606
25 0.852
0 1.0
input std: [0.97 1.01]
